# 第10回：LLM応用 (RAG・医学応用) 〜 信頼される AI を構築する 〜

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/DS_Seminar1_Public/blob/main/notebooks/session10_llm_applications.ipynb)

**DSゼミナールⅠ（2026年度）**  
熊本大学 データサイエンス学科

---

## 📋 本日の学習ロードマップ (計90分)

1. **【講義1】ハルシネーション：なぜ AI は自信満々に嘘をつくのか？ (10分)**
   - 確率的生成の限界と、医学教育におけるリスク。
2. **【講義2】RAG (検索拡張生成) の仕組み：持ち込み試験モデル (20分)**
   - ベクトルデータベース、埋め込み (Embedding)、コサイン類似度。
3. **【実習1】簡易 RAG システムの構築 (20分)**
   - 独自の知識ベースに基づいた正確な回答エンジンの作成。
4. **【講義3】ファインチューニングの最前線 (10分)**
   - 全学習 vs LoRA (低ランク適応)。少量の計算資源で専門家を作る手法。
5. **【講義4】AI エージェント：自律的に思考し、道具を使う AI (10分)**
   - ReAct モデル：推論 (Reasoning) と行動 (Acting) のループ。
6. **【実習2】PubMed API との連携シミュレーション (10分)**
   - 論文データベースを検索し、要約を生成するエージェントの流れ。
7. **【総合演習】「次世代 AI 医療助手」の設計 (10分)**

---

## 1. セットアップ

埋め込みモデル (`sentence-transformers`) や、ベクトル演算ライブラリを使用します。

In [ ]:
!pip install -q sentence-transformers sklearn scipy

import numpy as np
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine
import torch

print("RAG Implementation Engine, Warmed up.")

## 2. 【講義1+2】RAG：ハルシネーションへの唯一の処方箋

### 2.1 ハルシネーションの本質
LLM は、知識を「覚えている」のではなく、文脈から「もっともらしい続き」を確率で選んでいるだけです。そのため、存在しない論文名や副作用をスラスラと作り上げてしまいます。

### 2.2 RAG (Retrieval-Augmented Generation)
質問が来たとき、まず **「信頼できるデータベース」** から関連する資料を探し出し、その資料をプロンプト（指示文）の中に埋め込んで AI に渡します。
1. **Embed (埋め込み)**: 情報を高次元ベクトルの座標に変換する。
2. **Retrieve (検索)**: 質問のベクトルと「距離が近い」資料を見つける。
3. **Augment (拡張)**: 資料をプロンプトに追加する。
4. **Generate (生成)**: 資料に基づいて AI が答える。

---

## 3. 【実習1】ベクトル検索による RAG の核となる技術の体験

「埋め込み (Embedding)」を使って、意味が近い文章を AI に選ばせます。

In [ ]:
# 軽量な埋め込みモデルをロード
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# 私たちの知識データベース（各セッションの要約）
documents = [
    "Session 2 covers descriptive statistics like mean, median, and variance.",
    "Session 4 introduces supervised learning including regression and classification.",
    "Session 8 explains Convolutional Neural Networks for image recognition.",
    "Session 9 discusses Transformers and large language models."
]

# 1. ドキュメントをベクトル化して「データベース」に登録
doc_embeddings = embedder.encode(documents)

# 2. ユーザーの質問
query = "Tell me about how AI processes pictures."
query_embedding = embedder.encode([query])[0]

# 3. 類似度（コサイン類似度）を計算して、最も「近い」資料を探す
similarities = [1 - cosine(query_embedding, doc_emb) for doc_emb in doc_embeddings]
best_idx = np.argmax(similarities)

print(f"Query: {query}")
print(f"Retrieved Info: {documents[best_idx]} (Score: {similarities[best_idx]:.4f})")
print("\nこれを AI に渡せば、正確な回答が得られます！")

## 4. 【講義3+4】さらに高度な活用：エージェントと LoRA

### 4.1 LoRA (Low-Rank Adaptation)
巨大なモデルの全パラメータを書き換えるのではなく、一部に「小さな行列（バイパス）」を追加してそこだけ学習させる手法です。これにより、個人レベルの PC でも LLM を自分好みに調教できるようになりました。

### 4.2 AI エージェント：道具を使わせる
AI に「もし統計計算が必要なら Python を実行せよ」「もし論文が必要なら PubMed を検索せよ」という権限を与える手法です。AI は自ら考えて外部ツールを使い、複雑なタスクを完了させます。

---

## 5. 【実習2】AI エージェントの「思考プロセス」をトレースする

AI エージェントがどのように課題を解決するか、疑似コードでシミュレーションします。

In [ ]:
def mock_agent(query):
    print(f"[User Query]: {query}")
    print("[Thought]: The user is asking about the medical trend. I should search PubMed.")
    print("[Action]: Call API: PubMed_Search('Immunotherapy 2026')")
    print("[Observation]: Found 532 new papers. Most mention 'Personalized T-cell therapy'.")
    print("[Thought]: I have the information. Now I summarize it for the user.")
    print("[Final Answer]: 2026年のトレンドは、個別化されたT細胞療法です。理由は...")

mock_agent("最近の免疫療法のトレンドを教えて")

---

## ✏️ 本日の最終研究ミッション (AI Medical Assistant Architect)

**シナリオ**: あなたは、熊本大学病院専用の「論文検索・回答エージェント」の開発責任者です。

### 課題 1: 類似度検索の限界
「コサイン類似度」だけでは、言葉の表面的な似ているさ（例：Apple と Fruit）は捉えられても、論理的な否定（例：効果がある $vs$ 効果がない）の区別が難しいことがあります。これを解決するために、検索結果をさらに AI に精査させる **「リランキング (Reranking)」** という手法について調査し、その仕組みを 3 行で説明してください。

### 課題 2: 倫理とバイアスの壁
特定の地域や人種に偏った論文データベースだけを RAG に組み込んだ場合、生成されるアドバイスにはどのような「公平性の欠如」が生じる可能性がありますか？ 医療 AI の倫理的観点から考察してください。

### 課題 3: 未来の医療システム設計
「音声で診察を記録し、自動で RAG を通して過去のカルテと照らし合わせ、疑わしい病名を AI エージェントがサジェストする」というシステムを作る場合、セキュリティ面（個人情報保護）で最も注意すべき点は、技術的にどこにあると考えますか？ 本講座の知識を総動員して論じてください。

---

In [ ]:
# ここに回答を記述してください



---
**Last updated**: 2026-02-15
**Instructor**: Nakaura-T (DS Department, Kumamoto Univ)